In [3]:
import os
os.chdir('src/')
print(os.getcwd())

/work/jessica.mcdonald/newCM1_LETKF/src


In [4]:
path = "observations/8may24_cm1_obs.csv"
if os.path.exists(path) == False:
    new_path = os.path.join(os.getcwd().replace('src', 'run'), path)
    print(os.path.exists(new_path))

True


In [1]:
#import datetime as dt
import cftime 
from datetime import datetime as py_datetime

import time
import datetime
from datetime import datetime as py_datetime
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt

import sys
sys.path.append('src/')
sys.path.append('src/fsrc/')
import ens

from scipy.ndimage import gaussian_filter
from matplotlib.colors import LinearSegmentedColormap, to_rgb

# from contextlib import redirect_stdout
# import io
def findnearest(array, value):
    array = np.asarray(array)
    return (np.abs(array-value)).argmin()

## settings for plots - makes them look nicer
import matplotlib as mpl
import matplotlib.pyplot as plt
#mpl.rcParams.keys()
mpl.rcParams.update({"axes.grid" : False, "grid.color": "0.6",  'grid.linestyle':':',
                    'grid.linewidth':1, 'axes.labelweight':'bold', 'legend.framealpha':1.0, 'lines.solid_capstyle':'butt',
                     'axes.labelsize':13, 'axes.titlesize':18, 'ytick.labelsize':13, 'xtick.labelsize':13, 'axes.titleweight':'bold'})
ens2nr = {'UH_2-5km': {25: 46, 50: 94, 75: 141, 100: 182, 125: 226, 150: 268}, 
               'dbz': {30: 8, 35: 16, 40: 24, 45: 32, 50: 40, 55: 48}}

def rename(exp, version = 'RAW'):
    '''versions are FUll, H, V, CI, or RAW'''


    version = version.upper()

    if (exp!= 'Control') & (version !='RAW'):

        split = exp.split('_')
        e1 = split[0]
        e_extra = '' 
        
        H = int(split[1][2:])
        if   H == 7: e2 = 'H7.5'
        elif H == 9: e2 = 'H9.0'
        else:        e2 = f'H{H}'
        
        
        if len(split) == 3: 
            e3 = 'V4.5'
            eR = split[2]
            
        else:
            V = int(split[3][1:])
            if V == 2: e3 = 'V2.5'
            if V == 6: e3 = 'V6.5'
    
               
    
            if len(split) == 5:
                e_extra = f'_{split[4]}'
    
        if version=='FULL': name = f'{e1}_{e2}_{e3}'
        if version=='H':    name = f'{e1}_{e2}'
        if version=='V':    name = f'{e1}_{e3}'
    
        if version == 'RAW': name = exp
        if version == 'CI': name = e1
        if version == 'DENSITY' : name = f'{int(split[1][0])}km{e_extra}'
    else:
        if exp == 'Control': name = 'NoDA'
        else: name = exp
            

    return name

import skimage.measure as sm
from scipy import stats

import netCDF4 as ncdf
import numpy as N
import math as M
import glob
import sys
import os
import re

import matplotlib
import matplotlib.pyplot as P
from matplotlib import ticker
from matplotlib.offsetbox import AnchoredText

import json
from pyproj import Proj
from time import time as timer
from pyDart import dll_2_dxy, dxy_2_dll
import pyDart
import datetime as dt
from subprocess import *
#from cbook2 import *
import src.plot_src.ctables as ctables
from optparse import OptionParser
import scipy.interpolate
import scipy.ndimage as ndimage
import scipy.spatial

from mpl_toolkits.basemap import Basemap


import sys
sys.path.append('fsrc/')
from fpython2 import fstate, addbubbles_box, obs_2_grid3d
from fpython2 import add_smooth_perts
#import fsrc.recursive2d as recursive2d

import state_vector as state



#from ens import read_CM1_ens

def subplot_format(n):
    ''' keeps an approximately square subplot organization. 
    The longer number will always be columns (so the figure will be wider than tall). 
    Reverse row,col in the return if you would prefer a taller plot'''
    
    facs= list( set(x for tup in ([i, n//i] for i in range(1, int(n**0.5)+1) if n % i == 0) for x in tup))
    if len(facs) %2 == 0:
        row, col = facs[int(len(facs)/2)-1:int(len(facs)/2)+1]
    else:
        col = row = int(np.sqrt(n))
    return row, col

def silence_print(func, *args, **kwargs):
    with redirect_stdout(io.StringIO()) as f: # this just silences all the obnoxious output
        return func(*args, **kwargs)

#===============================================================================
from ens import mymap
import scipy
import pandas as pd
import colormaps as cmaps

def get_dbz_fast(analysis_time, exp, cref=True, ob_file=None):

    fpath = f'/work/jessica.mcdonald/CM1_LETKF_2025/experiments/{exp}/'
    
    with open(f'{fpath}/letkf.exp', 'rb') as f: exper = json.load(f)
    
    # These values are set in the experiment dictionary created at the begining of the run
    
    xoffset = exper['xoffset']
    yoffset = exper['yoffset']
    glat    = exper['lat0']
    glon    = exper['lon0']
      
    if ob_file == None:
      ob_file = exper['radar_obs']
    
    ob_f = pd.read_csv(ob_file)
    
    # look for reflectivity +/- 2.5 min 
    
    dt            = datetime.timedelta(0,30)
    begin  = cftime.date2num(analysis_time - dt, "seconds since 1970-01-01 00:00:00")
    ending = cftime.date2num(analysis_time + dt, "seconds since 1970-01-01 00:00:00")
    
    data_mask = (ob_f.utime >= begin) & (ob_f.utime <= ending) &(ob_f.kind==12) # only dbz
    
    odata = ob_f[data_mask]
    
    data    = odata['value'][:]
    lats    = odata['lat'][:]
    lons    = odata['lon'][:]
    hgts    = odata['height'][:]
    
    cm1 = exper['cm1namelist']
    names = [c[1] for c in cm1]
    nx=cm1[np.where(np.array(names)=='nx')[0][0]][2]
    ny=cm1[np.where(np.array(names)=='ny')[0][0]][2]
    nz=cm1[np.where(np.array(names)=='nz')[0][0]][2]
    dx=cm1[np.where(np.array(names)=='dx')[0][0]][2]
    
    fstate_xc = np.arange(0, nx*dx, dx) + dx/2
    fstate_xc += exper['xoffset']
    fstate_yc = np.arange(0, ny*dx, dx) + dx/2
    fstate_yc += exper['yoffset']
    
    temp = xr.open_dataset(sorted(glob.glob(f"{exper['fcst_members'][0]}/*.nc"))[0], decode_timedelta=True)
    fstate_zc=temp.zh.values
    temp.close()
    
    map      = mymap(fstate_xc, fstate_yc, glat, glon)
    xob, yob = map(lons, lats)
    xob, yob = xob+xoffset, yob+yoffset
    
    # The coordinate system here is based on the grid lat0/lon0/hgt0, and the offset grid stored in fstate
    #     ens stores the x/y grid in grid-internal coordinates
    # Create obs list for KDTree query...
    xyz_obs  = np.vstack((hgts,yob,xob))
    obs_list = list(xyz_obs.transpose())
    
    # Create 3D grid arrays for KDTree
    y_array, z_array, x_array = np.meshgrid(fstate_yc, fstate_zc, fstate_xc)
    xyz_grid = np.dstack([z_array.ravel(),y_array.ravel(),x_array.ravel()])[0]
      
    # Use cKDTree to create fast indexing for 3D grid....
    
    mytree = scipy.spatial.cKDTree(xyz_grid)
    distance, indices1D = mytree.query(obs_list)
    
    # these are the integer indices that you now pass into the fortran routine. They
    # are the un-raveled 3D index locations nearest the observation point in the 3D array
    
    kk,jj,ii = np.unravel_index(indices1D, (nz,ny,nx)) 
    dbz3d = obs_2_grid3d(data, xob, yob, hgts, x_array, y_array, z_array, ii, jj, kk, 4000., 2000., 0.0)

    # Create composite reflectivity, and then replicate it into a 3D array
    
    dbz2d = np.zeros((1,))
    
    if cref:
    
        dbz2d = dbz3d.max(axis=0)
    
        for k in np.arange(nz):
            dbz3d[k] = dbz2d
      
    #if verbose: print("\n ==> ens_GRID_REFL: 2D composite DBZ requested   Max:  %4.1f   Min:  %4.1f" % (dbz2d.max(), dbz2d.min()))
    
    del xyz_obs, x_array, y_array, z_array, xyz_grid, mytree
             
    return dbz3d


def get_mean_var(files, var, height=None, return_dims=False, return_all_dims=False):
    myvar = []
    for f in files:
        cm1 = xr.open_dataset(f, decode_timedelta=True)
        if height != None:
            zi = findnearest(cm1.zh, height)
            myvar.append(cm1[var][0,zi].values)
        else:
            myvar.append(cm1[var][0].values)
    if return_dims:
        z,y,x = cm1.zh.values, cm1.yh.values, cm1.xh.values
        
    cm1.close()
    meanvar = np.mean(myvar, axis=0)
    
    if return_dims:
        if (height==None) | (return_all_dims==True):
            return z,y,x, meanvar
        else:
            return y,x, meanvar
    else:
        return meanvar


def pmm(cm1, var, height,t, return_mean=False):
    ''' height is in km'''

    N = cm1.sizes['ne']-1
    zi = findnearest(cm1.z, height)

    ens_mean = cm1[var][t, -1,zi]
    ranked_ens =sorted(cm1[var][t, :-1, zi].values.ravel())[::N]
    
    x,y = np.meshgrid(cm1.xh, cm1.yh)
    d = {'ens':ens_mean.values.ravel(), 'x':x.ravel(), 'y':y.ravel()}
    df = pd.DataFrame(d)
    
    df_ranked = df.sort_values(by='ens')
    df_ranked['ens'] = ranked_ens
    
    df_pmm = df_ranked.sort_index()

    if return_mean:
    
        return df_pmm['ens'].values.reshape(np.shape(x)), ens_mean
    else:
        return df_pmm['ens'].values.reshape(np.shape(x))



    
def add_scale(ax, xstart=200, xlength=50, y=280, cap_length=200*.04, text_bump =0):

    xend = xstart+xlength
    
    ax.plot([xstart,xend], [y,y], color='k')
    ax.plot([xstart,xstart], [y-cap_length/2,y+cap_length/2], color='k')
    ax.plot([xend, xend], [y-cap_length/2,y+cap_length/2], color='k')
    #ax.text(xstart+xlength/2, y-cap_length/2, '{xlength} km', ha='center', va='top', fontsize=13)
    ax.text(xstart+xlength/2, text_bump+y+cap_length/2, f'{xlength} km', ha='center', va='bottom', fontsize=13)


def raw_probs(field, threshold, ensN):
   
    probs = np.zeros_like(field)
    probs[field >= threshold] = 1
    return probs.sum(axis=0)/ensN

def performance_diagram(ax, axis_labels=True, colorbar_auto=False, cax = None, return_colorbar_info = True):

    ax.set_aspect('equal')
    ax.set_xlim(0,1.)
    ax.set_ylim(0,1.)
    
    sr_array = np.linspace(0.001,1,200)
    pod_array = np.linspace(0.001,1,200)
    X,Y = np.meshgrid(sr_array,pod_array)
    csi_vals = (X** -1 + Y ** -1 - 1.) ** -1 
    im = ax.contourf(X,Y,csi_vals,levels=np.arange(0,1.1,0.1),cmap='Greys', alpha=0.3, antialiased=True) #blue_blgra2_r,
    ax.contour(X,Y,csi_vals, levels=[0.2, 0.5, 0.8], colors='0.5', alpha=0.3)

    fb = X/Y
    ax.contour(X,Y,fb,levels=[1],linestyles='--',colors='0.5', linewidths=1)
    bias = ax.contour(X,Y,fb,levels=[0.1,0.5,1,1.5,2,3,5,10],linestyles='--',colors='Grey', linewidths=0.5)
    plt.clabel(bias, inline=True, inline_spacing=2,fmt='%.1f', fontsize=12,colors='0.3')

    if colorbar_auto:
        cbar_ax = fig.add_axes([0.92, 0.2, 0.02, 0.6])
        cbar = fig.colorbar(im, cax=cbar_ax)
        cbar.set_label('Critical Success Index (CSI)', weight='bold')
        [cbar.ax.axhline(l, color='0.5', alpha=0.3)  for l in [0.2, 0.5, 0.8]]
    if cax != None:
        cbar = fig.colorbar(im, cax=cax)
        cbar.set_label('Critical Success Index (CSI)', weight='bold')
        [cbar.ax.axhline(l, color='0.5', alpha=0.3)  for l in [0.2, 0.5, 0.8]]
        
    if axis_labels:
        ax.set_xlabel('Success Ratio (1-FAR)')
        ax.set_ylabel('POD')

    if return_colorbar_info:
        return im

def open_exp(path, fcst_start=False):
    fpath = os.path.join(path ,'letkf.exp' )
    if os.path.exists(fpath):
        with open(fpath, 'rb') as f: exper = json.load(f)
        if fcst_start: f_start = dt.datetime.strptime(exper['DA_PARAMS']['DA_end_time'],'%Y-%m-%d %H:%M:%S')
    else:
        with open(os.path.join(path, 'fcst.exp' ), 'rb') as f: exper = json.load(f)
        if fcst_start: f_start = dt.datetime.strptime(exper['FORECAST']['start'],'%Y-%m-%d %H:%M:%S')

    if fcst_start:
        return exper, f_start
    else:
        return exper

def calc_csi(X, Y):
    return (X** -1 + Y ** -1 - 1.) ** -1 


In [2]:
fpaths = glob.glob('/work/jessica.mcdonald/CM1_LETKF_2025/experiments/*dense_fix/')
fpaths = glob.glob('/work/jessica.mcdonald/CM1_LETKF_2025/experiments/C*_*noAI/')


for fpath in fpaths:

    if os.path.isfile(f'{fpath}/full_ens_analysis.nc') != True:
        print(os.path.basename(fpath))

        
        
        with open(f'{fpath}/letkf.exp', 'rb') as f:exper = json.load(f)
        #print('PRODUCING FORECAST FILES')
        
        start = dt.datetime.strptime(exper['DA_PARAMS']['DA_start_time'], '%Y-%m-%d %H:%M:%S')
        end   = dt.datetime.strptime(exper['DA_PARAMS']['DA_end_time'], '%Y-%m-%d %H:%M:%S')
        freq  = dt.timedelta(seconds = exper['DA_PARAMS']['assim_freq'])
        
        timearr = np.arange(start, end+freq, freq)
        for i, time in enumerate(timearr):
            time = time.astype(dt.datetime)
        
            files, myDT = ens.FindRestartFiles(exper, time, ret_exp=False, prior=False)
            ens_state = ens.read_CM1_ens(files, exper, state_vector=None, DateTime=myDT, addmean=1) # the final member is the mean member!!!!
            ens.ens_CM1_coords(ens_state)
        
            if i == 0:
                allvars = ens_state['state_vector']['xyz3d']
        
                nt,nk,nj,ni,ne = len(timearr), ens_state['nz'],ens_state['ny'],ens_state['nx'],ens_state['ne']+1
                
                coords = {'t'    : np.arange(0, nt),'ni'   : np.arange(0, ni),'nj'   : np.arange(0, nj),'nk'   : np.arange(0, nk), 'ne':np.arange(0,ne)}
                data_vars = { 'time': (['t'], [t.astype(dt.datetime) for t in timearr]), 
                              'xh': (['ni'], ens_state['xc'][:]/1000), 
                              'yh': (['nj'], ens_state['yc'][:]/1000),
                              'z' : (['nk'], ens_state['zc'][:]/1000)}
                
                for v in allvars:
                    data_vars[v.lower()] = (['t','ne','nk','nj','ni'], np.zeros([nt,ne,nk,nj,ni])*np.nan)
                myds = xr.Dataset(data_vars, coords)
        
            for v in allvars:
                myds[v.lower()][i,:,:,:,:] = ens_state[v][:]
        
            del ens_state
        
        # now add in 2-5 km UH
        z = myds.z
        ztop = findnearest(z, 5)
        zbot = findnearest(z, 2)
        
        # note = we are calculating this for the mean member as well
        w        = myds['w'][:, :, zbot:ztop+1].values
        u        = myds['u'][:, :, zbot:ztop+1].values
        v        = myds['v'][:, :, zbot:ztop+1].values
        
        dx  = np.gradient(myds.xh)*1000
        dy  = np.gradient(myds.yh)*1000
        dz  = np.gradient(myds.z)[zbot:ztop+1]*1000
        
        dudy = np.gradient(u, axis=3)/dy
        dvdx = np.gradient(v, axis=4)/dx
        zeta = dvdx-dudy
        
        UH = np.sum(zeta*w*dz[np.newaxis, np.newaxis, :, np.newaxis, np.newaxis], axis=2)
        del u,v,dx,dy,dz,dudy, dvdx, zeta
        
        bs      = myds.th[:, :,0, :, -1].mean(axis=(2)).values
        thp     = myds.th[:, :,0].values - bs[:,:, np.newaxis, np.newaxis]
        cp_size = np.where(thp <= -1, 1, 0).sum(axis=(2,3))*(3*3)
        cp_mean = np.nanmean(np.where(thp<=-1, thp, np.nan), axis=(2,3))
        
        myds['cp_temp']  = (('t', 'ne'), cp_mean)
        myds['cp_size']  = (('t', 'ne'), cp_size)
        
        myds['UH_2-5km'] = (('t', 'ne','nj','ni'), UH)
        myds['w_2-5km']  = (('t', 'ne', 'nj', 'ni'), w.mean(axis=2))
        myds['cdbz']     = (('t', 'ne', 'nj', 'ni'), myds.dbz.max(axis=2).values) # add in composite reflectivity 
        
        del w
        
        myds.to_netcdf(exper['base_path']+'/full_ens_analysis.nc', format='NETCDF4')



 ==> FindRestartFiles: Date and time supplied is 2024_05_08 20:00:00

 ==> FindRestartFile:  Found time 1800 in file:  /work/jessica.mcdonald/CM1_LETKF_2025/experiments/C03_H18_V6_noAI/member001/cm1rst_000002.nc

 ==> READ_CM1_ENS:  No state_vector supplied - trying to match microphysics schemes

 ==> READ_CM1_ENS ==> Matching state_vector found !!! using zvdh

 ==> READ_CM1_ENS: reading from time INDEX:  0
 ==> READ_CM1_ENS: reading from state TIME:  2024-05-08 20:00:00 


 Wallclock time for serial read netCDF ensemble files: 23.113  sec

 Wallclock time to convert from C to A grid: 0.924  sec

 Wallclock time to create ensemble means: 1.533  sec

 Wallclock time to read arrays from netCDF ensemble files: 26.247  sec

 Wallclock time to create coordinates: 0.0  sec

 ==> FindRestartFiles: Date and time supplied is 2024_05_08 20:03:00

 ==> FindRestartFile:  Found time 1980 in file:  /work/jessica.mcdonald/CM1_LETKF_2025/experiments/C03_H18_V6_noAI/member001/cm1rst_000003.nc

 ==> R

KeyboardInterrupt: 